# 双向轮动与实际恢复动作净优势

一个预期负向机制、一个主指标；全日历、原有合格状态、固定标签与成本。无训练、无阈值搜索；仅为复用历史必要条件检验。

In [1]:
from pathlib import Path
import os,json,hashlib
import numpy as np
import pandas as pd
import qlib
from qlib.data import D
from qlib.workflow import R
from src.data.minute_intake import atomic_json
from src.data.nas_market_data import archive_nas_recorder
root=Path.cwd()
while not (root/'SPEC.md').exists():
 if root==root.parent:raise RuntimeError('repository directory required')
 root=root.parent
os.chdir(root);os.environ['MLFLOW_ALLOW_FILE_STORE']='true';card=root/'research/rotation_information_20261014';plan=json.loads((card/'plan.json').read_text());assert hashlib.sha256((root/plan['provider']/'qlib_store_manifest.json').read_bytes()).hexdigest()==plan['provider_manifest_sha256'];qlib.init(provider_uri=str(root/plan['provider']),region='us');calendar=pd.DatetimeIndex(D.calendar(start_time=plan['start'],end_time=plan['as_of']+' 16:00:00',freq='5min')).normalize().unique();assert len(calendar)==502
loaded_sources={}
def load(key,artifact):
 spec=plan['sources'][key];p=root/spec['card'];s=json.loads((p/'summary.json').read_text());mp=p/'verification_manifest.json';assert hashlib.sha256(mp.read_bytes()).hexdigest()==spec['manifest_sha256'];entry=next(e for e in json.loads(mp.read_text())['entries'] if e['path'].endswith('/artifacts/'+artifact))
 with R.uri_context(s['recorder_uri']):
  r=R.get_recorder(recorder_id=spec['recorder_id'],experiment_name=p.name);f=Path(r.download_artifact(artifact));assert hashlib.sha256(f.read_bytes()).hexdigest()==entry['sha256'];loaded_sources[key+'/'+artifact]=entry;return r.load_object(artifact)
feature=load('features','analysis/transitions').set_index('date');snap=load('features','analysis/snapshots');bounds=load('measurement','analysis/bounds');labels=load('labels','analysis/labels');eligibility=load('eligibility','analysis/daily')
base=feature.reindex(calendar)[['known','assumed_available_at','two_way_sign_rotation']].rename(columns={'two_way_sign_rotation':'score'});assert base.known.all();base.index.name='trade_date';base['year']=base.index.year
market=snap.pivot(index='date',columns='cutoff',values='soxx_return').reindex(calendar);assert market.notna().all().all();base['soxx_open_to_1030']=market['10:30'];base['soxx_0945_to_1030']=(1+market['10:30'])/(1+market['09:45'])-1;base['state']=base.soxx_open_to_1030.ge(0).astype(int)*2+base.soxx_0945_to_1030.ge(0).astype(int);assert (base.assumed_available_at==base.index+pd.Timedelta(hours=10,minutes=31)).all()
stress=bounds.loc[bounds.opening_halfwidth_usd.eq(.005)].set_index('date')[['lower','upper']];base=base.join(stress);base['measurement_zero_ambiguous']=base.lower.eq(0)&base.upper.gt(0)
def statistic(data):
 p=data.dropna(subset=['score','small_advantage']).reset_index(drop=True)
 if len(p)<2:return np.nan
 groups=['year','state'];rx=p.groupby(groups).score.rank(method='average',pct=True);ry=p.groupby(groups).small_advantage.rank(method='average',pct=True);rx=rx-rx.groupby([p.year,p.state]).transform('mean');ry=ry-ry.groupby([p.year,p.state]).transform('mean')
 if rx.std(ddof=0)==0 or ry.std(ddof=0)==0:return np.nan
 return float(rx.corr(ry))
frames={};intervals=[];annual=[];coverage=[];all_draws={}
for cost in plan['costs']:
 lab=labels.loc[labels.slippage_bps.eq(cost)].set_index('trade_date');assert lab.index.is_unique;assert lab.small_target_beta.le(.6+1e-12).all();frame=base.join(lab[['small_advantage','label_available_at','small_target_beta']]);ready=eligibility[str(cost)].set_index('trade_date').alpha_ready.reindex(calendar);assert ready.notna().all();frame['eligible']=ready
 observed=frame.small_advantage.notna();assert frame.loc[observed,'eligible'].all();assert (frame.loc[observed,'label_available_at']>frame.index[observed]+pd.Timedelta(hours=10,minutes=32)).all();mature=frame.label_available_at.notna()&frame.label_available_at.lt(pd.to_datetime((frame.year+1).astype(str)+'-01-01'));frame.loc[~mature,'small_advantage']=np.nan;frame['mature']=mature;frame['slippage_bps']=cost;frames[str(cost)]=frame
 for year,p in frame.groupby('year'):
  annual.append({'slippage_bps':cost,'year':year,'calendar_sessions':len(p),'eligible_sessions':int(p.eligible.sum()),'known_mature_labels':int(p.small_advantage.notna().sum()),'measurement_ambiguous_labeled_sessions':int((p.measurement_zero_ambiguous&p.small_advantage.notna()).sum()),'primary_estimate':statistic(p)})
 for (year,state),p in frame.groupby(['year','state']):coverage.append({'slippage_bps':cost,'year':year,'state':state,'calendar_sessions':len(p),'known_mature_labels':int(p.small_advantage.notna().sum()),'distinct_labeled_feature_values':int(p.loc[p.small_advantage.notna(),'score'].nunique())})
 point=statistic(frame);n=len(frame);rng=np.random.default_rng(plan['seed']);draws=[]
 for _ in range(plan['resamples']):
  ix=np.empty(n,dtype=int);ix[0]=rng.integers(n)
  for j in range(1,n):ix[j]=rng.integers(n) if rng.random()<1/plan['mean_block'] else (ix[j-1]+1)%n
  draws.append(statistic(frame.iloc[ix]))
 draws=np.asarray(draws);all_draws[str(cost)]=draws;valid=np.isfinite(draws);low,high=np.quantile(draws[valid],[.025,.975]) if valid.any() else [np.nan,np.nan];intervals.append({'slippage_bps':cost,'calendar_sessions':n,'known_mature_labels':int(frame.small_advantage.notna().sum()),'primary_estimate':point,'lower':low,'upper':high,'valid_draws':int(valid.sum())})
passed=all(np.isfinite(r['upper']) and r['upper']<0 for r in intervals) and all(np.isfinite(r['primary_estimate']) and r['primary_estimate']<0 for r in annual)
summary={'experiment':card.name,'decision':'necessary_condition_pass_requires_full_incremental_validation' if passed else 'not_promoted_fixed_rotation_information_failed','route_status':'await_full_incremental_validation' if passed else 'closed_no_sign_horizon_or_threshold_search','primary_gate_pass':bool(passed),'new_fits':0,'new_backtests':0,'new_orders':0,'objective_achieved':False,'provider_uri':str(root/plan['provider']),'region':'us','as_of':plan['as_of'],'scope':plan['scope'],'primary_metric':plan['primary_metric'],'expected_sign':plan['expected_sign'],'initial_cash':100000,'fee_model':'Frozen IBKR Tiered and independent 2/5bp same-state net local action labels','qlib_docs':'https://qlib.readthedocs.io/en/latest/component/recorder.html','artifact_policy':{'policy':'slim_card_recorder_first','workflow_config':'workflow.yaml','recorder_only_artifacts':['inputs/*','analysis/*']}}
tracking=card/'qlib_recorder'
with R.start(experiment_name=card.name,recorder_name='fixed_rotation_recovery_information',uri=str(tracking)):
 rec=R.get_recorder();rec.log_params(new_fits=0,new_backtests=0,expected_sign='negative',mean_block=plan['mean_block'],resamples=plan['resamples'])
 rec.save_objects(artifact_path='inputs',plan=plan,source_entries=loaded_sources)
 rec.save_objects(artifact_path='analysis',frames=frames,bootstrap=all_draws,annual=pd.DataFrame(annual),intervals=pd.DataFrame(intervals),control_coverage=pd.DataFrame(coverage),summary=summary)
 artifacts=tracking/str(rec.experiment_id)/rec.id/'artifacts';rec.log_metrics(registered_artifact_count=sum(p.is_file() for p in artifacts.rglob('*')))
summary.update(recorder_id=rec.id,experiment_id=str(rec.experiment_id),recorder_uri=str(tracking));atomic_json({'recorder_id':rec.id,'entries':[{'path':str(p.relative_to(card)),'sha256':hashlib.sha256(p.read_bytes()).hexdigest()} for p in sorted(artifacts.rglob('*')) if p.is_file()]},card/'verification_manifest.json');summary['nas_evidence']=archive_nas_recorder(card/'nas_binding.json',workspace_root=root,card=card,recorder_id=rec.id,experiment_id=str(rec.experiment_id));atomic_json(summary,card/'summary.json');pd.DataFrame(annual).to_csv(card/'annual_information.csv',index=False);pd.DataFrame(intervals).to_csv(card/'information_intervals.csv',index=False);pd.DataFrame(coverage).to_csv(card/'control_coverage.csv',index=False)
import yaml
(card/'workflow.yaml').write_text(yaml.safe_dump({'experiment':card.name,'provider_uri':summary['provider_uri'],'recorder_uri':summary['recorder_uri'],'artifact_policy':summary['artifact_policy']},sort_keys=False));assert hashlib.sha256((root/'production/stage1_futu_sim_v1/runtime_release.json').read_bytes()).hexdigest()==plan['release_sha256'];print(pd.DataFrame(annual).to_string(index=False));print(pd.DataFrame(intervals).to_string(index=False))


# 双向轮动未能区分恢复动作净优势，固定解释关闭

本轮0训练、0新增回测、0阈值搜索、0订单。固定2024—2025全部502个原生日历位置，复用10万美元、IBKR Tiered、2/5bp各自独立状态的恢复/HOLD局部净优势标签，动作beta上限0.6不变。入场10:32，下一09:47同一参考目标共同重置；未来重置目标只用于事后标签，不作为特征。该标签不是完整策略收益，也不是新的独立样本外。

特征仍为原two_way_sign_rotation，09:45与10:30同一开盘基准的严格双向跨零权重，研究可用时点10:31。预期方向在读取本轮标签关系前固定为负：轮动高意味着内部修复不一致，恢复动作的净优势较低。未改成负特征、未使用测量下界代替原值、未按测量稳定性过滤日期。

主指标按年度及ETF两种事前方向组合分组：开盘至10:30涨跌，09:45至10:30涨跌，零归入非负。组内分别计算特征与局部优势的百分位秩并中心化，随后计算相关。它只控制共同市场方向，不充分控制幅度、波动、旧模型全部输入或账户状态，故只是固定必要条件诊断，不能称为完整特征增量比较。

|成本|年份|全部日历|原合格状态|年内成熟标签|条件排序关联|
|---|---|---|---|---|---|
|2bp|2024|252|66|66|+0.05113|
|2bp|2025|250|42|42|+0.00037|
|5bp|2024|252|107|107|+0.06167|
|5bp|2025|250|197|196|+0.05159|

|成本|两年成熟标签|主指标|95%区块区间|
|---|---|---|---|
|2bp|108|+0.03139|[-0.13995, +0.19456]|
|5bp|303|+0.05507|[-0.04636, +0.14834]|

四个年度/成本点估计均非预期负向，两个合并区间均跨零。固定通过条件要求两个成本区间上界均小于0、四个年度方向均为负，因此失败。区间保留全部502个交易位置，平均63日的stationary block、1000次重采样，每次重新计算组内秩；两个成本各1000次有效。未知/不合格日没有压缩成连续交易日；年末未成熟标签保留未知，5bp在2025年有1个此类状态。

两种成本的资格日和账户状态不同，不能把5bp与2bp的样本数或点估计差异解释成提高成本有益。测量零值歧义的成熟标签，2bp两年有2/1日，5bp有4/8日，全部保留；不剔除这些日期寻找较好结果。区块区间没有校正整个项目此前多轮研究选择，且有效标签较少，所以既不能证明独立超额，也不能以区间跨零断言真实作用严格为零。

决定：not_promoted_fixed_rotation_information_failed；这条“更高双向轮动对应更低局部恢复净优势”的固定解释关闭，不翻号、不换期限、不搜索特征/动作阈值、不追加拟合救活。输入可计算、来源精度多数稳定，与对未来净优势有信息是不同门槛。本轮没有理由将该特征加入现役模型或据此减少恢复动作。

稳定净SOXX超额仍未实现；完整策略MaxDD/CVaR、年度/滚动稳定性及合格前向门均未因本轮获得通过。现役release保持不变。

下一项信息需求必须来自一个不同、可解释且在决策前可得的机制，并能说明相对已有价格、波动和内部广度究竟新增什么；不能把继续增加成员、改广度定义或换频率当作信息增量。本路线不再追加检验；后续独立机制仍须先冻结唯一主问题与停止规则，才能进入完整旧信息对照和净费账户。

[年度结果](annual_information.csv) · [区块区间](information_intervals.csv) · [各控制组覆盖](control_coverage.csv) · [固定方案](plan.json)

Recorder `63f3c48b0b63403ea6859ad5c5165b9c`。
